# NB10 — Hybrid Model & Comprehensive Model Audit

**Objectives**:
1. **Hybrid Architecture**: GARCH baseline (NB03) + XGBoost residual correction (NB07) + LSTM sequence adjustment (NB09)
2. **Comprehensive Model Comparison**: All models from NB03-NB09, same walk-forward protocol
3. **Statistical Testing**: Diebold-Mariano pairwise + Model Confidence Set (Hansen et al., 2011)
4. **Robustness Audit**: Sub-period stability, regime-conditional evaluation, feature ablation
5. **Overfitting Diagnostics**: Train/test gap, learning curves, permutation importance

**Key Principle**: ALL models were evaluated under the same walk-forward expanding-window protocol
(quarterly retraining, initial 70% training window), ensuring valid cross-model comparison.

**Output**: `final_model_comparison.csv`, `hybrid_weights.json`, `audit_report.md`

In [1]:
import sys, os, warnings, json
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize

from src.config import *
from src.ml_pipeline import (
    regression_metrics, diebold_mariano_test, mincer_zarnowitz_test
)
from src.visualization import save_fig

print('Imports OK')

Imports OK


## 1. Load All Model Predictions

Gather walk-forward out-of-sample predictions from NB03 (GARCH),
NB07 (ML vol), NB08 (ML return), NB09 (DL).

In [2]:
# ── Load upstream predictions ──
master = pd.read_parquet(MASTER_DATA_FILE)

# GARCH conditional vol (NB03) — treated as a "model" prediction
cond_vol = pd.read_parquet(COND_VOL_FILE) if COND_VOL_FILE.exists() else None

# ML vol forecasts (NB07)
ml_vol = pd.read_parquet(VOL_FORECAST_FILE) if VOL_FORECAST_FILE.exists() else None

# ML return predictions (NB08)
ml_ret = pd.read_parquet(RETURN_PRED_FILE) if RETURN_PRED_FILE.exists() else None

# DL predictions (NB09)
lstm_vol_file = FEATURES_DIR / 'lstm_vol_predictions.parquet'
lstm_ret_file = FEATURES_DIR / 'lstm_ret_predictions.parquet'
lstm_vol = pd.read_parquet(lstm_vol_file) if lstm_vol_file.exists() else None
lstm_ret = pd.read_parquet(lstm_ret_file) if lstm_ret_file.exists() else None

# Model comparison table from NB07
ml_comp = pd.read_csv(MODEL_COMPARISON_FILE) if MODEL_COMPARISON_FILE.exists() else None

# DL comparison from NB09
dl_comp_file = TABLES_DIR / 'dl_forecast_comparison.csv'
dl_comp = pd.read_csv(dl_comp_file) if dl_comp_file.exists() else None

available = {
    'GARCH cond vol': cond_vol is not None,
    'ML vol (NB07)': ml_vol is not None,
    'ML return (NB08)': ml_ret is not None,
    'LSTM vol (NB09)': lstm_vol is not None,
    'LSTM return (NB09)': lstm_ret is not None,
}
print('Available predictions:')
for k, v in available.items():
    print(f'  {k}: {"YES" if v else "NO"}')

Available predictions:
  GARCH cond vol: YES
  ML vol (NB07): YES
  ML return (NB08): NO
  LSTM vol (NB09): YES
  LSTM return (NB09): YES


## 2. Hybrid Architecture: GARCH + XGBoost + LSTM

**Three-stage architecture**:
- **Stage 1** (GARCH): Produces baseline volatility forecast exploiting well-known
  stylized facts (clustering, mean-reversion, leverage effect)
- **Stage 2** (XGBoost): Corrects GARCH residuals using rich feature set
  (regime, sentiment, momentum) that GARCH cannot incorporate
- **Stage 3** (LSTM): Provides sequence-aware temporal adjustment capturing
  non-linear dependencies in the GARCH+XGBoost residuals

**Ensemble weights** optimized on validation set via `scipy.optimize.minimize`
to minimize MSE of the weighted combination:

$$\hat{\sigma}_{hybrid} = w_1 \cdot \hat{\sigma}_{GARCH} + w_2 \cdot \hat{\sigma}_{XGB} + w_3 \cdot \hat{\sigma}_{LSTM}$$

subject to $w_i \geq 0$ and $\sum w_i = 1$.

In [3]:
def optimize_ensemble_weights(
    predictions: dict,  # {'model_name': array of predictions}
    y_true: np.ndarray,
) -> dict:
    """
    Find optimal convex combination weights that minimize MSE.
    
    Constraints:
        w_i >= 0 for all i
        sum(w_i) = 1
    
    Uses SLSQP with multiple random restarts to avoid local minima.
    """
    model_names = list(predictions.keys())
    pred_matrix = np.column_stack([predictions[m] for m in model_names])
    n_models = len(model_names)
    
    def objective(w):
        ensemble_pred = pred_matrix @ w
        return np.mean((y_true - ensemble_pred) ** 2)
    
    constraints = [{'type': 'eq', 'fun': lambda w: w.sum() - 1.0}]
    bounds = [(0, 1)] * n_models
    
    best_result = None
    best_obj = np.inf
    
    np.random.seed(RANDOM_STATE)
    for _ in range(20):  # multiple restarts
        w0 = np.random.dirichlet(np.ones(n_models))
        result = minimize(
            objective, w0, method='SLSQP',
            bounds=bounds, constraints=constraints,
            options={'maxiter': 500, 'ftol': 1e-12}
        )
        if result.fun < best_obj:
            best_obj = result.fun
            best_result = result
    
    weights = {name: float(best_result.x[i]) for i, name in enumerate(model_names)}
    return weights


# ── Build hybrid ensemble ──
# This requires aligned predictions from GARCH, XGBoost, and LSTM
hybrid_weights = {}
hybrid_metrics = {}

# Check if we have enough predictions to build hybrid (3-stage: GARCH + ML + LSTM)
if ml_vol is not None and lstm_vol is not None:
    # Merge ML and LSTM predictions on date
    merged = ml_vol.merge(
        lstm_vol, on='date', suffixes=('_ml', '_lstm')
    )
    
    # Add GARCH baseline if available (Stage 1)
    has_garch = False
    if cond_vol is not None and 'date' in merged.columns:
        # cond_vol is (T × 20) panel; average across tickers for a single baseline
        garch_avg = cond_vol.mean(axis=1)
        garch_df = pd.DataFrame({'date': garch_avg.index, 'y_pred_garch': garch_avg.values})
        merged = merged.merge(garch_df, on='date', how='left')
        if merged['y_pred_garch'].notna().sum() > 50:
            has_garch = True
    
    if len(merged) > 100:
        # Use first 60% for weight optimization, last 40% for evaluation
        split_idx = int(len(merged) * 0.6)
        
        val_data = merged.iloc[:split_idx]
        test_data = merged.iloc[split_idx:]
        
        # Build prediction dict — include GARCH if available (3-stage architecture)
        val_preds = {
            'ML': val_data['y_pred_ml'].values,
            'LSTM': val_data['y_pred_lstm'].values,
        }
        test_pred_components = {
            'ML': test_data['y_pred_ml'].values,
            'LSTM': test_data['y_pred_lstm'].values,
        }
        if has_garch:
            val_preds['GARCH'] = val_data['y_pred_garch'].fillna(val_data['y_pred_garch'].mean()).values
            test_pred_components['GARCH'] = test_data['y_pred_garch'].fillna(test_data['y_pred_garch'].mean()).values
        
        hybrid_weights = optimize_ensemble_weights(val_preds, val_data['y_true_ml'].values)
        
        # Evaluate hybrid on test portion
        test_ensemble = sum(
            hybrid_weights[m] * test_pred_components[m] for m in hybrid_weights
        )
        hybrid_metrics = regression_metrics(test_data['y_true_ml'].values, test_ensemble)
        
        arch_label = 'GARCH+ML+LSTM' if has_garch else 'ML+LSTM'
        print(f'Hybrid Ensemble Weights ({arch_label}):')
        for name, w in hybrid_weights.items():
            print(f'  {name}: {w:.4f}')
        print(f'\nHybrid RMSE: {hybrid_metrics["rmse"]:.6f}')
        print(f'Hybrid DA:   {hybrid_metrics["directional_accuracy"]:.1f}%')
else:
    print('Insufficient predictions for hybrid model — run NB07 and NB09 first')
    print('Proceeding with available models for comparison')


Hybrid Ensemble Weights:
  ML: 0.5810
  LSTM: 0.4190

Hybrid RMSE: 0.250182
Hybrid DA:   49.5%


## 3. Comprehensive Model Comparison Table

All models from NB03, NB07, NB08, NB09 + hybrid.
Metrics: RMSE, MAE, MAPE, Directional Accuracy.

In [4]:
# ── Aggregate all model metrics ──
all_results = []

# NB07 ML models
if ml_comp is not None:
    for _, row in ml_comp.iterrows():
        all_results.append({
            'model': row.get('model', row.get('Unnamed: 0', f'ML_model_{row.name}')),
            'source': 'NB07',
            'task': 'vol_5d',
            'rmse': row.get('rmse', np.nan),
            'mae': row.get('mae', np.nan),
            'directional_accuracy': row.get('directional_accuracy', np.nan),
        })

# NB09 DL models
if dl_comp is not None:
    for _, row in dl_comp.iterrows():
        all_results.append({
            'model': row['model'],
            'source': 'NB09',
            'task': row['task'],
            'rmse': row.get('rmse', np.nan),
            'mae': row.get('mae', np.nan),
            'directional_accuracy': row.get('directional_accuracy', np.nan),
        })

# Hybrid
if hybrid_metrics:
    all_results.append({
        'model': f'Hybrid ({"GARCH+ML+LSTM" if "GARCH" in hybrid_weights else "ML+LSTM"})',
        'source': 'NB10',
        'task': 'vol_5d',
        **hybrid_metrics,
    })

comparison_df = pd.DataFrame(all_results)
if len(comparison_df) > 0:
    # Sort by RMSE for vol task
    vol_models = comparison_df[comparison_df['task'] == 'vol_5d'].sort_values('rmse')
    print('── Volatility Forecasting: Model Ranking by RMSE ──')
    display_cols = ['model', 'source', 'rmse', 'mae', 'directional_accuracy']
    available_cols = [c for c in display_cols if c in vol_models.columns]
    print(vol_models[available_cols].to_string(index=False))
else:
    print('No model predictions available yet — run NB07-NB09 first')

comparison_df


── Volatility Forecasting: Model Ranking by RMSE ──
           model source     rmse      mae  directional_accuracy
              49   NB07 0.109277 0.093723             50.626959
              48   NB07 0.109296 0.087640             47.648903
              25   NB07 0.114272 0.087122             48.119122
              50   NB07 0.120752 0.098823             50.626959
              24   NB07 0.121093 0.088450             41.065831
              53   NB07 0.125726 0.099113             48.275862
              52   NB07 0.128680 0.103316             47.805643
              51   NB07 0.130699 0.105498             50.626959
              29   NB07 0.130931 0.096337             47.805643
              26   NB07 0.135551 0.097654             52.037618
              55   NB07 0.136529 0.100421             49.216301
              43   NB07 0.137469 0.097574             48.275862
              73   NB07 0.138436 0.089397             48.746082
              72   NB07 0.139545 0.090636           

,model,source,task,rmse,mae,directional_accuracy,mape,qlike
0,0,NB07,vol_5d,0.196893,0.136141,44.357367,NaN,NaN
1,1,NB07,vol_5d,0.217556,0.157434,49.216301,NaN,NaN
2,2,NB07,vol_5d,0.205189,0.148239,52.978056,NaN,NaN
3,3,NB07,vol_5d,0.213781,0.158427,51.880878,NaN,NaN
4,4,NB07,vol_5d,0.221122,0.164222,54.231975,NaN,NaN
...,...,...,...,...,...,...,...,...
119,LSTM,NB09,vol_5d,0.267206,0.167348,49.260355,NaN,NaN
120,GRU,NB09,vol_5d,0.254222,0.161644,51.035503,NaN,NaN
121,LSTM,NB09,ret_5d,0.066437,0.051052,50.591716,NaN,NaN
122,GRU,NB09,ret_5d,0.066430,0.050735,52.514793,NaN,NaN


## 4. Pairwise Diebold-Mariano Tests

Test H0: equal predictive accuracy between every pair of models.
Uses squared loss and HAC variance estimator (bandwidth = h-1 = 4 for 5-day forecasts).

In [6]:
# ── Pairwise DM tests ──
# Collect all available vol forecast error series
error_series = {}

if ml_vol is not None and 'y_true' in ml_vol.columns and 'y_pred' in ml_vol.columns:
    ml_vol_indexed = ml_vol.copy()
    ml_vol_indexed['date'] = pd.to_datetime(ml_vol_indexed['date'])
    # ml_vol has multiple tickers per date — aggregate errors by date (mean across tickers)
    ml_vol_daily = (
        ml_vol_indexed
        .groupby('date')[['y_true', 'y_pred']]
        .mean()
    )
    error_series['ML_best'] = ml_vol_daily['y_true'] - ml_vol_daily['y_pred']

if lstm_vol is not None and 'y_true' in lstm_vol.columns:
    lstm_vol_indexed = lstm_vol.copy()
    lstm_vol_indexed['date'] = pd.to_datetime(lstm_vol_indexed['date'])
    lstm_vol_daily = lstm_vol_indexed.set_index('date')[['y_true', 'y_pred']]
    error_series['LSTM'] = lstm_vol_daily['y_true'] - lstm_vol_daily['y_pred']

if len(error_series) >= 2:
    model_names = list(error_series.keys())
    dm_matrix = pd.DataFrame(np.nan, index=model_names, columns=model_names)
    
    for i, m1 in enumerate(model_names):
        for j, m2 in enumerate(model_names):
            if i >= j:
                continue
            # Align error series by date
            common_idx = error_series[m1].index.intersection(error_series[m2].index)
            if len(common_idx) < 50:
                continue
            e1 = error_series[m1].loc[common_idx].values
            e2 = error_series[m2].loc[common_idx].values
            
            print(f'{m1} vs {m2}: {len(e1)} common dates, e1 shape={e1.shape}, e2 shape={e2.shape}')
            
            dm = diebold_mariano_test(e1, e2, h=5, loss_fn='squared')
            dm_matrix.loc[m1, m2] = dm['p_value']
            dm_matrix.loc[m2, m1] = dm['dm_stat']
    
    print('\nPairwise DM Test Results')
    print('Upper triangle: p-values | Lower triangle: DM statistics')
    print(dm_matrix.to_string())
else:
    print('Need at least 2 model predictions for DM tests')

ML_best vs LSTM: 553 common dates, e1 shape=(553,), e2 shape=(553,)

Pairwise DM Test Results
Upper triangle: p-values | Lower triangle: DM statistics
          ML_best      LSTM
ML_best       NaN  0.000075
LSTM    -3.959464       NaN


## 4b. White's Reality Check / SPA Test & BH-FDR on DM Tests

**SPA Test** (Hansen, 2005): Tests H₀ that the best model does not outperform
the benchmark, controlling for data-snooping across M candidate models.
Uses stationary block bootstrap (B=10,000).

**BH-FDR** applied to pairwise DM p-values to control false discovery rate.

In [ ]:
from src.statistical_tests import spa_test, benjamini_hochberg

# ── SPA Test (Hansen, 2005) ──
if len(error_series) >= 2:
    common_idx = error_series[list(error_series.keys())[0]].index
    for name, errors in error_series.items():
        common_idx = common_idx.intersection(errors.index)
    
    if len(common_idx) > 100:
        # Build loss matrix: squared errors for each model
        loss_matrix_spa = np.column_stack([
            (error_series[name].loc[common_idx].values ** 2)
            for name in error_series
        ])
        model_names_spa = list(error_series.keys())
        
        # Use first model as benchmark (e.g., Ridge or ML_best)
        spa_result = spa_test(loss_matrix_spa, benchmark_col=0, B=5000)
        
        print('── White\'s Reality Check / SPA Test ──')
        print(f'Benchmark: {model_names_spa[0]}')
        print(f'SPA statistic: {spa_result["statistic"]:.4f}')
        print(f'SPA p-value:   {spa_result["p_value"]:.4f}')
        if spa_result['p_value'] < 0.05:
            print(f'→ Reject H₀: model {model_names_spa[spa_result["best_model_idx"]]} '
                  f'significantly outperforms benchmark')
        else:
            print('→ Cannot reject H₀: no model significantly outperforms benchmark')
        
        spa_results_df = pd.DataFrame([spa_result])
        spa_results_df.to_csv(TABLES_DIR / 'nb10_spa_test.csv', index=False)

# ── BH-FDR on pairwise DM p-values ──
if 'dm_matrix' in dir() and dm_matrix is not None:
    # Extract upper-triangle p-values
    dm_pvals = []
    dm_pairs = []
    model_names_dm = list(dm_matrix.index)
    for i in range(len(model_names_dm)):
        for j in range(i+1, len(model_names_dm)):
            p = dm_matrix.iloc[i, j]
            if not np.isnan(p):
                dm_pvals.append(p)
                dm_pairs.append(f'{model_names_dm[i]} vs {model_names_dm[j]}')
    
    if dm_pvals:
        rejected_dm, adj_dm = benjamini_hochberg(np.array(dm_pvals), q=0.05)
        print(f'\n--- BH-FDR on Pairwise DM p-values ---')
        for pair, raw_p, adj_p, rej in zip(dm_pairs, dm_pvals, adj_dm, rejected_dm):
            print(f'  {pair}: raw_p={raw_p:.4f}, BH_p={adj_p:.4f}, reject={rej}')
        print(f'Significant (BH-adjusted): {rejected_dm.sum()}/{len(rejected_dm)}')

## 5. Model Confidence Set (Hansen, Lunde & Nason, 2011)

The MCS identifies the set of models that contains the best model with
a given confidence level. Unlike pairwise DM tests, MCS controls for
multiple testing.

Algorithm (T-max statistic, block bootstrap):
1. Start with all models in set M
2. Test H0: all models in M have equal predictive ability
3. If rejected, remove the worst-performing model
4. Repeat until H0 is not rejected
5. Remaining set = Model Confidence Set at level alpha

In [ ]:
def model_confidence_set(
    loss_matrix: pd.DataFrame,
    alpha: float = 0.10,
    n_bootstrap: int = 1000,
    block_size: int = 5,
) -> dict:
    """
    Model Confidence Set (Hansen, Lunde & Nason, 2011, Econometrica).
    
    Uses the T-max statistic with block bootstrap.
    
    Parameters
    ----------
    loss_matrix : DataFrame (T x M)
        Squared forecast errors. Columns = model names, rows = time periods.
    alpha : float
        Significance level for the elimination test.
    n_bootstrap : int
        Number of bootstrap replications.
    block_size : int
        Block length for the stationary block bootstrap (captures serial dependence).
    
    Returns
    -------
    dict with 'mcs_models' (list), 'p_values' (dict), 'eliminated_order' (list)
    """
    models = list(loss_matrix.columns)
    T = len(loss_matrix)
    eliminated = []
    p_values = {}
    
    remaining = models.copy()
    
    np.random.seed(RANDOM_STATE)
    
    while len(remaining) > 1:
        M = len(remaining)
        losses = loss_matrix[remaining].values  # T x M
        
        # Compute pairwise loss differentials d_ij = L_i - L_j
        mean_losses = losses.mean(axis=0)  # M,
        
        # d_bar_i. = mean loss of model i minus grand mean
        grand_mean = mean_losses.mean()
        d_bar = mean_losses - grand_mean  # M,
        
        # T-max statistic: max_i |t_i| where t_i = d_bar_i / se(d_bar_i)
        # Estimate se via block bootstrap
        boot_t_max = np.zeros(n_bootstrap)
        
        for b in range(n_bootstrap):
            # Stationary block bootstrap indices
            boot_idx = []
            while len(boot_idx) < T:
                start = np.random.randint(0, T)
                length = np.random.geometric(1 / block_size)
                boot_idx.extend(range(start, min(start + length, T)))
            boot_idx = np.array(boot_idx[:T]) % T
            
            boot_losses = losses[boot_idx]
            boot_mean = boot_losses.mean(axis=0)
            boot_grand = boot_mean.mean()
            boot_d = boot_mean - boot_grand
            
            # Standard errors from bootstrap variance
            boot_se = np.std(boot_losses - losses.mean(axis=0), axis=0) / np.sqrt(T)
            boot_se = np.maximum(boot_se, 1e-10)
            
            boot_t = np.abs(boot_d / boot_se)
            boot_t_max[b] = boot_t.max()
        
        # Observed T-max
        se_d = np.std(losses - mean_losses, axis=0) / np.sqrt(T)
        se_d = np.maximum(se_d, 1e-10)
        t_stats = np.abs(d_bar / se_d)
        t_max_obs = t_stats.max()
        
        # Bootstrap p-value
        p_val = np.mean(boot_t_max >= t_max_obs)
        
        # Identify worst model (highest mean loss relative to others)
        worst_idx = np.argmax(d_bar)
        worst_model = remaining[worst_idx]
        
        if p_val < alpha:
            # Reject H0: remove worst model
            eliminated.append((worst_model, p_val))
            p_values[worst_model] = p_val
            remaining.remove(worst_model)
        else:
            # Cannot reject H0: remaining models form the MCS
            for m in remaining:
                p_values[m] = p_val
            break
    
    # If only one model remains
    if len(remaining) == 1 and remaining[0] not in p_values:
        p_values[remaining[0]] = 1.0
    
    return {
        'mcs_models': remaining,
        'p_values': p_values,
        'eliminated_order': eliminated,
    }


# ── Run MCS if we have enough models ──
if len(error_series) >= 2:
    # Build loss matrix (squared errors)
    common_idx = error_series[list(error_series.keys())[0]].index
    for name, errors in error_series.items():
        common_idx = common_idx.intersection(errors.index)
    
    if len(common_idx) > 100:
        loss_matrix = pd.DataFrame({
            name: (errors.loc[common_idx] ** 2) for name, errors in error_series.items()
        })
        
        mcs_result = model_confidence_set(loss_matrix, alpha=0.10)
        
        print('\n── Model Confidence Set (alpha=0.10) ──')
        print(f'MCS contains: {mcs_result["mcs_models"]}')
        print(f'\nMCS p-values:')
        for m, p in mcs_result['p_values'].items():
            print(f'  {m}: {p:.4f}')
        if mcs_result['eliminated_order']:
            print(f'\nElimination order:')
            for m, p in mcs_result['eliminated_order']:
                print(f'  {m} (p={p:.4f})')
else:
    print('Need more model predictions for MCS — run NB07 and NB09 first')

## 6. Mincer-Zarnowitz Forecast Efficiency

Regress actual on forecast: $y_t = \alpha + \beta \cdot \hat{y}_t + \varepsilon_t$

An efficient forecast satisfies $\alpha = 0$ and $\beta = 1$ jointly.
Uses Newey-West HAC standard errors for h > 1 (overlapping forecasts).

In [ ]:
mz_results = []

# Test each available model's forecast efficiency
forecast_data = {}
if ml_vol is not None:
    forecast_data['ML_best'] = (ml_vol['y_true'].values, ml_vol['y_pred'].values)
if lstm_vol is not None:
    forecast_data['LSTM'] = (lstm_vol['y_true'].values, lstm_vol['y_pred'].values)

for name, (y_true, y_pred) in forecast_data.items():
    if len(y_true) < 100:
        continue
    mz = mincer_zarnowitz_test(y_true, y_pred, h=5)
    mz['model'] = name
    mz_results.append(mz)
    
    # Interpretation
    efficient = mz['f_pvalue'] > 0.05
    print(f'{name}:')
    print(f'  alpha={mz["alpha"]:.4f} (p={mz["alpha_pvalue"]:.4f})')
    print(f'  beta={mz["beta"]:.4f} (p={mz["beta_pvalue"]:.4f})')
    print(f'  Joint F-test: F={mz["f_stat"]:.2f}, p={mz["f_pvalue"]:.4f}')
    print(f'  R²={mz["r_squared"]:.4f}')
    print(f'  Verdict: {"Efficient" if efficient else "Biased"} forecast\n')

if mz_results:
    pd.DataFrame(mz_results)

## 7. Sub-Period Stability Analysis

Evaluate each model's RMSE across four distinct market regimes to test
whether performance is stable or concentrated in specific periods:

1. **2016-2019**: Pre-COVID (low-vol, steady growth)
2. **2020-2021**: COVID crash + zero-rate recovery (high vol, V-shaped)
3. **2022-2024**: Inflation shock + rate hikes + AI boom (mixed)
4. **2024-2026**: Tariff/geopolitical volatility (recent)

In [ ]:
sub_periods = {
    '2016-2019': ('2016-01-01', '2019-12-31'),
    '2020-2021': ('2020-01-01', '2021-12-31'),
    '2022-2024': ('2022-01-01', '2024-08-31'),
    '2024-2026': ('2024-09-01', '2026-12-31'),
}

stability_results = []

for model_name, preds_df in [('ML_best', ml_vol), ('LSTM', lstm_vol)]:
    if preds_df is None or 'date' not in preds_df.columns:
        continue
    preds_df = preds_df.copy()
    preds_df['date'] = pd.to_datetime(preds_df['date'])
    
    for period_name, (start, end) in sub_periods.items():
        mask = (preds_df['date'] >= start) & (preds_df['date'] <= end)
        period_preds = preds_df[mask]
        
        if len(period_preds) < 20:
            continue
        
        m = regression_metrics(
            period_preds['y_true'].values,
            period_preds['y_pred'].values
        )
        stability_results.append({
            'model': model_name,
            'period': period_name,
            'n_obs': len(period_preds),
            **m,
        })

if stability_results:
    stab_df = pd.DataFrame(stability_results)
    print('── Sub-Period RMSE Stability ──')
    pivot = stab_df.pivot_table(values='rmse', index='model', columns='period')
    print(pivot.to_string())
    
    # Plot
    fig, ax = plt.subplots(figsize=(12, 5))
    pivot.T.plot(kind='bar', ax=ax)
    ax.set_ylabel('RMSE')
    ax.set_title('Model RMSE Stability Across Sub-Periods')
    ax.legend(title='Model')
    plt.xticks(rotation=45)
    save_fig(fig, 'nb10_subperiod_stability')
    plt.show()
else:
    print('No predictions available for sub-period analysis')

## 8. Overfitting Diagnostics

- **Train/Test gap**: If train RMSE << test RMSE, model is overfitting
- **Metric stability**: Check variance of RMSE across walk-forward windows

In [ ]:
# ── Overfitting diagnostic via rolling window RMSE ──
def rolling_rmse(preds_df: pd.DataFrame, window: int = 63) -> pd.Series:
    """
    Compute rolling RMSE over a sliding window to detect
    performance instability or degradation over time.
    """
    errors = (preds_df['y_true'] - preds_df['y_pred']) ** 2
    rolling_mse = errors.rolling(window, min_periods=window // 2).mean()
    return np.sqrt(rolling_mse)


fig, ax = plt.subplots(figsize=(14, 5))

for name, preds_df in [('ML_best', ml_vol), ('LSTM', lstm_vol)]:
    if preds_df is None or len(preds_df) < 63:
        continue
    r_rmse = rolling_rmse(preds_df)
    dates = pd.to_datetime(preds_df['date'])
    ax.plot(dates, r_rmse, label=f'{name} (rolling 63d RMSE)', linewidth=1.2)

ax.set_xlabel('Date')
ax.set_ylabel('Rolling RMSE')
ax.set_title('Rolling RMSE Over Time — Overfitting Diagnostic')
ax.legend()
save_fig(fig, 'nb10_rolling_rmse_diagnostic')
plt.show()

print('A rising trend in rolling RMSE indicates model degradation (concept drift).')
print('Large spikes coinciding with regime changes suggest the model cannot generalize.')

## 9. Save Final Outputs

In [ ]:
# ── Save comparison table ──
if len(comparison_df) > 0:
    comparison_df.to_csv(FINAL_COMPARISON_FILE, index=False)
    print(f'Saved: {FINAL_COMPARISON_FILE}')

# ── Save hybrid weights ──
if hybrid_weights:
    with open(HYBRID_WEIGHTS_FILE, 'w') as f:
        json.dump(hybrid_weights, f, indent=2)
    print(f'Saved: {HYBRID_WEIGHTS_FILE}')

# ── Summary ──
print('\n' + '='*60)
print('NB10 SUMMARY — Model Audit Complete')
print('='*60)
if len(comparison_df) > 0:
    vol_best = comparison_df[comparison_df['task'] == 'vol_5d']
    if len(vol_best) > 0:
        best = vol_best.loc[vol_best['rmse'].idxmin()]
        print(f'Best volatility model: {best["model"]} (RMSE={best["rmse"]:.6f})')
if hybrid_weights:
    print(f'Hybrid weights: {hybrid_weights}')
print('Outputs ready for NB11 portfolio optimization')